In [1]:
# --------------------------------------------------------
# 
# Compare results from two feedstocks across application
# rate and particle size dimensions. 
# 
# Uses the `postproc_flxs` subdir to get the CDR inputs.
# 
# (this script replaces `cdr_cc-sil_compare-*` which was 
#  made before the postproc_flxs subdir code was created)
# 
# --------------------------------------------------------

In [2]:
# --- CDR calc note 
# --- from Yoshi (Jan 17, 2024)
# (-CO2_dif_exp-CO2_rsp_exp)-(-CO2_dif_spn-CO2_rsp_spn) - 0.14*(CO2_adv_exp - CO2_adv_spn).
# In the above equation CO2_dif denotes face values in the "diff" column for int_flux_gas-pco2.txt file,  
# CO2_adv in the column of "adv", CO2_rsp in the "g2" column (OM phase considered).  
# "exp" denotes ERW experiments while spn controls/spin-ups.

In [3]:
import fnmatch
import glob
import os
import pickle
import sys
import re
from typing import Tuple

import cmocean.cm as cmo
import fsspec         # for AWS integration
import s3fs
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D  # for custom legend entries (needed for contour plot)
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# --- read in cdr calculation functions  
sys.path.append(os.path.abspath('/home/tykukla/ew-workflows/run_scepter'))
import cdr_fxns_postproc as cfp
import sa_postproc_fxns as spf
# ---



In [4]:
# --- Decide which CDR calculations to perform
cdr_calc_list = ["co2_flx",       
                 "camg_flx",
                 "totcat_flx",
                 # "carbalk_flx", # (turn this back on when the runs are re-run with fixed cflx file)
                 "rockdiss"]
# ---

In [16]:

# --- 
runtype = "field"         # field or lab
fertlevel = "SA"          # "no", "low", "mid", or "hi"
dustsp = "cc"           # "gbas", "wls" (silicate species)
sitename = "311a"         # "311a" or "311b"
dur = '5'                 # [yr], for file naming
timehorizon_cdr = 1       # [yr], for integrating cdr signal

tag_name = f"morris_8_450"  # an extra tag in the casenames, something like "wet" (use "" for none)
version_csv = "v0"   # version appended to the csv file for the batch input

# --- SAVE DETAILS -----------------------------------------------------------------------------------------
save_results = True
# savedir_pref = f"meanAnn_shortRun_{fertlevel}Fert_{tag_sil}_multiyear"   # prefix for the save directory (which will be created upon save)
savedir_pref = f"meanAnn_shortRun_{fertlevel}Fert_{dustsp}_{tag_name}"   # prefix for the save directory (which will be created upon save)

savepath = "/home/tykukla/ew-workflows/scripts/scepter/process/runs/batch_postproc_tmp/sa_morris"
# ----------------------------------------------------------------------------------------------------------

# [ define the sensitivity vars used ]
sa_params = ["dustrate", "taudust", "dustrad", "qrun", "mat", "dust_mixdep", "dustrate_2nd"]

# [ set up ctrl and csv fn ]
ctrl_run_idx = 0    # the `make_batch_input_SA_v0.py` file forces the control run to the zero index
csv_fn = f"meanAnn_{dustsp}_shortRun_{tag_name}_{fertlevel}Fert_gs+apprate_{version_csv}.csv"


# ---
# groundwork
# outdir = "/home/tykukla/SCEPTER/scepter_output"
outdir = "s3://carbonplan-carbon-removal/SCEPTER/scepter_output_scratch/"
csv_loc = "/home/tykukla/ew-workflows/inputs/scepter/batch"

In [17]:
# --- read in the batch .csv
dfin = pd.read_csv(os.path.join(csv_loc, csv_fn))

# add column for the full run id
dfin["newrun_id_full"] = dfin['newrun_id'] + "_" + dfin['dustsp'] + "_" + runtype + "_tau"+dfin["duration"].astype(float).astype(str).str.replace(".", "p")  # (duration has to be turned into float first because otherwise we miss the decimal pt)
# identify the ctrl run
dfin['ctrl_run'] = dfin.index == dfin.index[0]
# eliminate other sites if necessary
if len(dfin['site'].unique()) > 1:
    dfin = dfin[dfin['site'] == f"site_{sitename}"].copy()
# add a column for the dustrate in ton_ha_yr
if "dustrate" in dfin.columns:
    dfin["dustrate_ton_ha_yr"] = dfin["dustrate"] / 100 


In [18]:
# # --- OPTIONAL: Filter out part of CSV

# # [1] by climatefile name (corresponds with site name)
# site = "site_311a"
# dfin_cc = dfin_cc[dfin_cc['climatefiles'] == 'site_311a']
# dfin_sil = dfin_sil[dfin_sil['climatefiles'] == 'site_311a']

# # [2] ... TK

In [19]:
# --- decide which columns from dfin we want to keep when we 
#     create our flux dicts below
# 
# these should be the columns that become dimensions in the 
# later xr datasets... 
# 
dfin_cols_to_keep = sa_params + ['dustrate_ton_ha_yr']

In [20]:
# --- read in the postprocessed flux data 
flx_dict = cfp.read_postproc_flux(dfin, outdir, cdr_calc_list, dfin_cols_to_keep, rockdiss_feedstock=dustsp)

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


In [21]:
# plt.scatter(flx_dict_sil['co2_flx']['time'], flx_dict_sil['co2_flx']['co2flx_dif'])
# flx_dict_sil['co2_flx'][flx_dict_sil['co2_flx']['ctrl'] == False]

In [22]:
# --- compute CDR fluxes for cc and sil (with losses considered)
# set the time_horizon for computing the integrated flux 
time_horizon = timehorizon_cdr # float(dur)
# ---
cdr_dict_full, cdr_dict_sum = cfp.cdr_int_per_group(flx_dict, time_horizon, cdr_calc_list, dfin_cols_to_keep)

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


In [23]:
cdr_dict_sum['co2_flx']

,units,flx_type,dustrate,taudust,dustrad,qrun,mat,dust_mixdep,dustrate_2nd,dustrate_ton_ha_yr,cdr_dif_component,cdr_resp_component,cdr_resp_component_noAE,cdr_dif,cdr_adv,cdr_adv_plus_newSIC,cdr_SIConly,tot_adv,time_horizon,cdr_fxn
0,ton ha-1,int_flx,1336.666667,0.030000,200.000000,1.033333,30.0,0.400000,11.666667,13.367,0.454807,-0.000814,-0.000814,0.453993,0.309933,0.309933,-1.110223e-16,0.771745,1,co2_flx
1,ton ha-1,int_flx,1336.666667,0.030000,200.000000,1.033333,30.0,0.400000,11.666667,13.367,0.454817,-0.000820,-0.000820,0.453997,0.309934,0.309934,1.110223e-16,0.771754,1,co2_flx
2,ton ha-1,int_flx,10.000000,0.030000,200.000000,1.033333,30.0,0.400000,11.666667,0.100,0.059066,0.000338,0.000000,0.059066,0.050172,0.050172,0.000000e+00,0.094111,1,co2_flx
3,ton ha-1,int_flx,10.000000,0.030000,200.000000,1.033333,30.0,0.400000,35.000000,0.100,0.051366,0.000354,0.000000,0.051366,0.042489,0.042489,8.326673e-17,0.086458,1,co2_flx
4,ton ha-1,int_flx,10.000000,0.030000,200.000000,1.033333,30.0,0.166667,35.000000,0.100,0.048781,0.000370,0.000000,0.048781,0.039920,0.039920,-2.775558e-17,0.083888,1,co2_flx
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,ton ha-1,int_flx,2000.000000,0.030000,10.000000,0.100000,30.0,0.166667,35.000000,20.000,-0.300533,-0.001367,-0.001367,-0.301900,-0.390928,-0.390928,0.000000e+00,-0.152156,1,co2_flx
446,ton ha-1,int_flx,673.333333,0.030000,10.000000,0.100000,30.0,0.166667,35.000000,6.733,-0.300905,-0.000071,-0.000071,-0.300976,-0.382606,-0.382606,5.551115e-17,-0.153662,1,co2_flx
447,ton ha-1,int_flx,673.333333,0.030000,10.000000,1.033333,30.0,0.166667,35.000000,6.733,0.288045,-0.000272,-0.000272,0.287773,0.151777,0.151777,2.775558e-17,0.702558,1,co2_flx
448,ton ha-1,int_flx,673.333333,0.076667,10.000000,1.033333,30.0,0.166667,35.000000,6.733,0.281930,-0.000307,-0.000307,0.281623,0.145808,0.145808,-2.775558e-17,0.690716,1,co2_flx


In [24]:
# --- get emissions 
dustrate_name = "dustrate_ton_ha_yr" # note, we multiply by duration in emissions_calc, so we only want annual rate here
# set the inputs ***************
p80_input = 1.3e3
truck_km = 0.1e3
barge_km = 0.0e3
barge_diesel_km = 0
Efactor_org = "MRO"
# ******************************

# []
cdr_dict_sum['rockdiss'] = cfp.emissions_calculator_df(cdr_dict_sum['rockdiss'], dustrate_name, 
                                                        p80_input, truck_km, barge_km, barge_diesel_km, 
                                                        Efactor_org, mineral=dustsp)

In [25]:
# -------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------

In [26]:
# --- save results **************************************************************************
# 
if save_results:
    # create the dictionary we'll save as the resource file
    savedict = spf.create_save_dict_sa(
                    cdr_calc_list,
                    group_vars = dfin_cols_to_keep,
                    csv_fn = csv_fn,
                    multiyear = False, 
                    time_horizon = time_horizon, 
                    cf_apprate_fixed = None,
                    cf_dustrad_fixed = None, 
                    p80_input = p80_input,
                    truck_km = truck_km,
                    barge_km = barge_km, 
                    barge_diesel_km = barge_diesel_km, 
                    Efactor_org = Efactor_org,
                )
    
    # write the files
    fname_res = "_calc_inputs.res"    # name of the resource file we'll save
    savedir_pref = f"{savedir_pref}_{str(int(time_horizon))}yrInt"
    saved_here = spf.save_batch_postproc_sa(
        savedict, 
        base_path = savepath,
        base_dir_name = savedir_pref,
        fname_res = fname_res,
        cdr_dict_full = cdr_dict_full,
        cdr_dict_sum = cdr_dict_sum,
        save_directory = None, # (use base path and base_dir_name)
    )
# *********************************************************************************************    

Directory created: /home/tykukla/ew-workflows/scripts/scepter/process/runs/batch_postproc_tmp/sa_morris/meanAnn_shortRun_SAFert_cc_morris_8_450_1yrInt_001


## PROFILE POSTPROCESSING:

In [21]:
# # --- prepare the post-processing inputs
# # 
# batch_axes = dfin_cols_to_keep
# #
# # dictionary for which files to process in the batch profile functions
# proc_dict = {
#     "adsorbed": False,
#     "adsorbed_percCEC": True,
#     "adsorbed_ppm": False,
#     "aqueous": True,
#     "aqueous_total": False,
#     "bulksoil": True,
#     "exchange_total": False,
#     "gas": True,
#     "rate": False,
#     "soil_ph": True,
#     "solid": False,
#     "solid_sp_saturation": True, 
#     "solid_volumePercent": True,
#     "solid_weightPercent": True,
#     "specific_surface_area": True,
#     "surface_area": False,
# }


In [44]:
# # --- compile all the profile .nc data into a single dataset
# #

# # [silicate]
# dsdict_sil = cfp.prof_batchprocess_allvars(outdir, dustsp, dfin_sil, batch_axes, proc_dict, print_loop_updates = False)

compiling profile data for fo


In [46]:
# --- save the postprocessed results
# 
# [silicate]
# cfp.save_batch_postproc_profOnly(dsdict_sil, dustsp, save_directory = saved_here)


'/home/tykukla/ew-workflows/scripts/scepter/process/runs/batch_postproc_tmp/cc-sil_psize_apprate/meanAnn_shortRun_lowFert_base_fo_001'

# -------------------------------------------------------------
# -------------------------------------------------------------